# 01 — OpenTracy Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenTracy/opentracy/blob/main/notebooks/01_quickstart.ipynb)

By the end of this notebook — **in under three minutes** — you'll have:

1. Installed the OpenTracy SDK
2. Made a real LLM call and seen `cost` + `latency` on the response
3. Switched providers with a one-string change
4. Added automatic fallbacks for production

No server, no Docker, no config files. Just an API key.

> **What you need:** an OpenAI API key. The provider swap section also uses Anthropic and Groq, but they're optional — skip those cells if you only have OpenAI.

## Install

One line. The wheel bundles the Go routing engine and ONNX embedder; nothing else to set up.

In [ ]:
%pip install opentracy -q --upgrade  

## Set your API key

We use `getpass` so the key isn't echoed into the notebook output.

In [7]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

## 1. Your first call — 30 seconds

Same shape as the OpenAI SDK — but with `_cost` and `_latency_ms` on the response automatically.

In [ ]:
# ---- OpenTracy platform hook-up (optional but usually what you want) ----
# By default, opentracy.completion() and opentracy.Router go DIRECT to the
# provider (OpenAI, Anthropic, etc.). That means traces, cost accounting,
# routing decisions, and the distillation loop will NOT see your calls.
#
# To make every call pass through the running OpenTracy engine so it
# shows up in the dashboard, point OPENTRACY_ENGINE_URL at the engine's
# HTTP API (default port 8080). The dashboard UI on :3000 is a different
# service (nginx) and will return 405 for POST /v1/chat/completions — do
# NOT use that URL here. Uncomment the two lines below BEFORE the
# `import opentracy` call in the next cell.
#
# Provider API keys are configured once via the UI (Settings → API Keys)
# or POST /v1/secrets/<provider>; the engine reads them from
# ~/.opentracy/secrets.json inside its container on every call.

# import os
# os.environ["OPENTRACY_ENGINE_URL"] = "http://localhost:8080"  # engine API (:8080), not dashboard UI (:3000)


In [9]:
import opentracy as ot

resp = ot.completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello in three words."}],
)

print(resp.choices[0].message.content)
print(f"cost:    ${resp._cost:.6f}")
print(f"latency: {resp._latency_ms:.0f}ms")
print(f"tokens:  {resp.usage.prompt_tokens} in / {resp.usage.completion_tokens} out")

Hello, how are you?
cost:    $0.000006
latency: 1530ms
tokens:  13 in / 6 out


**That's the hook.** Every response already carries `_cost` and `_latency_ms` — you didn't wire up any observability. Standard OpenAI fields (`choices`, `usage`, `.message.content`) also work as-is.

Useful extras on `resp`:

| Field | Meaning |
| --- | --- |
| `resp._cost` | USD for this one call (float) |
| `resp._latency_ms` | End-to-end latency in milliseconds |
| `resp._provider` | Which provider answered (`"openai"`, `"anthropic"`, ...) |
| `resp._routing` | Routing metadata if the engine was involved |

## 2. Switch providers with one string — 1 minute

Same function, same message shape, different provider. No new SDK, no new auth code.

Skip the cells below if you don't have keys for Anthropic / Groq — this is just to demonstrate the pattern.

In [ ]:
# Anthropic — uncomment and set ANTHROPIC_API_KEY to run
# if not os.environ.get("ANTHROPIC_API_KEY"):
#     os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")
#
# resp = ot.completion(
#     model="anthropic/claude-haiku-4-5-20251001",
#     messages=[{"role": "user", "content": "Say hello in three words."}],
# )
# print(resp.choices[0].message.content)
# print(f"cost: ${resp._cost:.6f}  provider: {resp._provider}")

In [ ]:
# Groq (fast Llama) — uncomment and set GROQ_API_KEY to run
# if not os.environ.get("GROQ_API_KEY"):
#     os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")
#
# resp = ot.completion(
#     model="groq/llama-3.3-70b-versatile",
#     messages=[{"role": "user", "content": "Say hello in three words."}],
# )
# print(resp.choices[0].message.content)
# print(f"cost: ${resp._cost:.6f}  latency: {resp._latency_ms:.0f}ms")

OpenTracy supports 13 providers total: OpenAI, Anthropic, Gemini, Groq, Mistral, DeepSeek, Together, Fireworks, Cerebras, Sambanova, Perplexity, Cohere, Bedrock. The only convention to learn is `provider/model`.

## 3. Add fallbacks — 1 minute

Production calls that survive one provider being down. If the primary fails (rate limit, outage, timeout), OpenTracy tries each fallback in order.

In [ ]:
resp = ot.completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Draft a pithy tagline for a coffee shop."}],
    fallbacks=[
        # "anthropic/claude-haiku-4-5-20251001",   # uncomment if you have the key
        # "groq/llama-3.3-70b-versatile",
    ],
    num_retries=1,
)

print(resp.choices[0].message.content)
print(f"answered by: {resp._provider}")

## 4. Streaming — 30 seconds

Token-by-token streaming works unchanged across every provider. OpenTracy translates Anthropic / Bedrock event-streams into OpenAI SSE, so your client code stays the same.

In [ ]:
stream = ot.completion(
    model="openai/gpt-4o-mini",
    messages=[{"role": "user", "content": "Count from 1 to 5, one number per line."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

## What you now have

- ✅ OpenAI-compatible completions with cost & latency metadata
- ✅ 13 providers behind one API
- ✅ Fallbacks for resilience
- ✅ Streaming across providers

## Next notebooks

| # | Notebook | What you'll learn |
| --- | --- | --- |
| 02 | **Drop-in over the OpenAI SDK** | Migrate existing OpenAI code with one line change |
| 03 | **Semantic routing** | Let the router pick the cheapest model that's good enough per prompt |
| 04 | **Ticket classifier (end-to-end)** | A real 50-line app showing before/after cost |
| 05 | **Distillation** | Train your own tiny student model from traffic (needs GPU) |

More at the [OpenTracy docs](https://docs.opentracy.ai).